<!-- paired-role-banner -->
> 🟧 **실습 노트북 06 · 빈 코드 창**
>
> 오른쪽의 같은 이름 **정답 노트북**과 셀 위치를 맞춰 보세요. 먼저 정답을 가리고 직접
> 작성한 뒤 막힐 때 한 줄씩 비교하는 방식을 권장합니다. `NotImplementedError`가 있는 셀을
> 완성하면 됩니다.

# 06. PyTorch 텍스트 분류 파이프라인

고객지원 티켓을 예제로 **데이터 로딩부터 오분류 분석까지** 직접 구현합니다. 프레임워크가 숨겨 주는 단계를 의도적으로 펼쳐 놓았기 때문에, 각 tensor의 shape를 추적하며 실행하세요.

예상 시간: 90~120분 · CUDA → MPS → CPU 자동 선택 · 결과물: 작은 분류 checkpoint

## 전체 흐름

`CSV → group-safe split → tokenizer → vocabulary → Dataset → collate/padding → DataLoader → model → loss/backward → checkpoint → error analysis`

현업에서 특히 중요한 두 가지를 함께 연습합니다.

- 같은 사건의 변형 문장이 train/validation 양쪽에 들어가는 **데이터 누수 방지**
- 모델 가중치뿐 아니라 vocabulary와 label mapping까지 저장하는 **재현 가능한 checkpoint**

In [ ]:
from collections import Counter
from dataclasses import dataclass, asdict
from pathlib import Path
import random
import re

import pandas as pd
import torch
from sklearn.metrics import f1_score
from torch import nn
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, Dataset
from llm_engineering_lab.acceleration import get_accelerator

SEED = 42
random.seed(SEED)
torch.manual_seed(SEED)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device

PROJECT_ROOT = next(
    (p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()),
    None,
)
assert PROJECT_ROOT is not None, "프로젝트 루트에서 JupyterLab을 실행해 주세요."
DATA_PATH = PROJECT_ROOT / "data" / "customer_support_tickets.csv"
print(ACCELERATOR.summary())
print("data:", DATA_PATH)

In [ ]:
tickets = pd.read_csv(DATA_PATH)
required = {"ticket_id", "case_id", "text", "label"}
missing = required - set(tickets.columns)
assert not missing, f"필수 열 누락: {missing}"
assert tickets["ticket_id"].is_unique
assert tickets[["case_id", "text", "label"]].notna().all().all()

print("shape:", tickets.shape)
print("labels:", sorted(tickets["label"].unique()))
tickets[["ticket_id", "case_id", "text", "label"]].head(3)

In [ ]:
label_summary = (
    tickets.groupby("label")
    .agg(rows=("ticket_id", "size"), cases=("case_id", "nunique"))
    .sort_index()
)
lengths = tickets["text"].str.len()
print(label_summary)
print("\n문자 길이 요약:")
print(lengths.describe(percentiles=[0.5, 0.9]).round(1))

# 한 case_id에 서로 다른 label이 섞이면 group split 전에 수정해야 합니다.
labels_per_case = tickets.groupby("case_id")["label"].nunique()
assert labels_per_case.max() == 1

## 1. Group-safe train/validation split

이 데이터에는 한 사건(`case_id`)을 다르게 표현한 문장이 여러 개 있습니다. 행을 무작위 분할하면 거의 같은 문장이 양쪽에 들어가 검증 점수가 부풀 수 있습니다.

따라서 label별 case 목록을 나눈 뒤 해당 case의 모든 행을 한쪽에 배치합니다.

In [ ]:
rng = random.Random(SEED)
validation_cases = set()

for label, group in tickets.groupby("label"):
    case_ids = sorted(group["case_id"].unique())
    rng.shuffle(case_ids)
    valid_count = max(1, round(len(case_ids) * 0.2))
    validation_cases.update(case_ids[:valid_count])

is_validation = tickets["case_id"].isin(validation_cases)
train_df = tickets.loc[~is_validation].reset_index(drop=True)
valid_df = tickets.loc[is_validation].reset_index(drop=True)

assert set(train_df["case_id"]).isdisjoint(valid_df["case_id"])
assert set(train_df["label"]) == set(valid_df["label"])
print("train/valid rows:", len(train_df), len(valid_df))
print("train/valid cases:", train_df.case_id.nunique(), valid_df.case_id.nunique())

## 2. 투명한 tokenizer

교육용으로 한글은 음절 단위, 영문/숫자는 묶음 단위로 나눕니다. 실무의 subword tokenizer보다 단순하지만 OOV, truncation, padding 개념을 관찰하기 좋습니다.

tokenizer는 train 데이터만 보고 vocabulary를 만들어야 합니다. validation을 포함해 vocabulary를 만들면 작은 형태의 데이터 누수가 됩니다.

In [ ]:
# TODO 1: 2. 투명한 tokenizer
# 오른쪽 정답 창의 같은 셀을 참고하되, 먼저 아래 구현을 직접 작성하세요.
# 1) 입력과 출력의 shape/dtype을 종이에 적습니다.
# 2) 최소 구현을 작성하고 Shift+Enter로 실행합니다.
# 3) 정답과 비교한 뒤 값 하나를 바꾸고 결과를 설명합니다.
raise NotImplementedError("06번 실습의 TODO 1을 완성하세요")

In [ ]:
# TODO 2: 2. 투명한 tokenizer
# 오른쪽 정답 창의 같은 셀을 참고하되, 먼저 아래 구현을 직접 작성하세요.
# 1) 입력과 출력의 shape/dtype을 종이에 적습니다.
# 2) 최소 구현을 작성하고 Shift+Enter로 실행합니다.
# 3) 정답과 비교한 뒤 값 하나를 바꾸고 결과를 설명합니다.
raise NotImplementedError("06번 실습의 TODO 2을 완성하세요")

In [ ]:
vocabulary = Vocabulary.build(train_df["text"], min_frequency=1)
encoded = vocabulary.encode("배송 API 오류 999")

print("vocab size:", len(vocabulary))
print("special ids:", {token: vocabulary.stoi[token] for token in SPECIAL_TOKENS})
print("encoded:", encoded)
print("decoded:", vocabulary.decode(encoded))
print("OOV count:", encoded.count(vocabulary.stoi["<unk>"]))
assert vocabulary.stoi["<pad>"] == 0

## 3. Dataset: 한 샘플을 tensor로 바꾸기

`Dataset.__getitem__`은 한 샘플만 반환합니다. 아직 padding하지 않는 이유는 배치마다 가장 긴 길이에 맞춰 padding해야 불필요한 계산을 줄일 수 있기 때문입니다.

In [ ]:
# TODO 3: 3. Dataset: 한 샘플을 tensor로 바꾸기
# 오른쪽 정답 창의 같은 셀을 참고하되, 먼저 아래 구현을 직접 작성하세요.
# 1) 입력과 출력의 shape/dtype을 종이에 적습니다.
# 2) 최소 구현을 작성하고 Shift+Enter로 실행합니다.
# 3) 정답과 비교한 뒤 값 하나를 바꾸고 결과를 설명합니다.
raise NotImplementedError("06번 실습의 TODO 3을 완성하세요")

In [ ]:
item = train_dataset[0]
print("keys:", item.keys())
print("input shape/dtype:", item["input_ids"].shape, item["input_ids"].dtype)
print("label:", item["label"].item(), id_to_label[item["label"].item()])
print("text:", item["text"])
assert item["input_ids"].ndim == 1
assert item["label"].ndim == 0

## 4. Collate: 가변 길이를 한 배치로 묶기

`DataLoader`가 샘플 목록을 `collate_fn`에 전달합니다. 여기서 `[T]` 여러 개를 `[B,T_max]`로 padding하고, 실제 토큰 위치가 1인 `attention_mask`를 만듭니다.

In [ ]:
# TODO 4: 4. Collate: 가변 길이를 한 배치로 묶기
# 오른쪽 정답 창의 같은 셀을 참고하되, 먼저 아래 구현을 직접 작성하세요.
# 1) 입력과 출력의 shape/dtype을 종이에 적습니다.
# 2) 최소 구현을 작성하고 Shift+Enter로 실행합니다.
# 3) 정답과 비교한 뒤 값 하나를 바꾸고 결과를 설명합니다.
raise NotImplementedError("06번 실습의 TODO 4을 완성하세요")

## 5. 분류 모델

구조는 `Embedding → masked mean pooling → LayerNorm → MLP → logits`입니다. 단순하지만 LLM 파이프라인의 입력 ID, mask, embedding, logits 개념을 그대로 포함합니다.

In [ ]:
@dataclass
class ClassifierConfig:
    vocab_size: int
    num_classes: int
    pad_id: int
    embedding_dim: int = 32
    hidden_dim: int = 48
    dropout: float = 0.15


class MeanPoolClassifier(nn.Module):
    def __init__(self, config: ClassifierConfig):
        super().__init__()
        self.config = config
        self.embedding = nn.Embedding(
            config.vocab_size, config.embedding_dim, padding_idx=config.pad_id
        )
        self.head = nn.Sequential(
            nn.LayerNorm(config.embedding_dim),
            nn.Linear(config.embedding_dim, config.hidden_dim),
            nn.GELU(),
            nn.Dropout(config.dropout),
            nn.Linear(config.hidden_dim, config.num_classes),
        )

    def forward(self, input_ids, attention_mask):
        assert input_ids.ndim == attention_mask.ndim == 2
        x = self.embedding(input_ids)  # [B, T, D]
        weights = attention_mask.unsqueeze(-1).to(x.dtype)
        pooled = (x * weights).sum(1) / weights.sum(1).clamp_min(1.0)
        logits = self.head(pooled)
        assert logits.shape == (input_ids.shape[0], self.config.num_classes)
        return logits

In [ ]:
config = ClassifierConfig(len(vocabulary), len(label_names), PAD_ID)
model = MeanPoolClassifier(config).to(DEVICE)
logits = model(demo_batch["input_ids"], demo_batch["attention_mask"])
probabilities = logits.softmax(dim=-1)

print("logits:", logits.shape)
print("first probabilities:", probabilities[0])
print("probability sum:", probabilities[0].sum().item())
assert torch.allclose(
    probabilities.sum(-1), torch.ones_like(probabilities.sum(-1)), atol=1e-6
)

In [ ]:
# 한 번의 update를 분해해 확인합니다.
one_step_model = MeanPoolClassifier(config).to(DEVICE)
one_step_optimizer = torch.optim.AdamW(one_step_model.parameters(), lr=3e-3)
before = one_step_model.head[-1].weight.detach().clone()

one_step_optimizer.zero_grad(set_to_none=True)
one_step_logits = one_step_model(demo_batch["input_ids"], demo_batch["attention_mask"])
one_step_loss = nn.functional.cross_entropy(one_step_logits, demo_batch["labels"])
one_step_loss.backward()
gradient_norm = nn.utils.clip_grad_norm_(one_step_model.parameters(), max_norm=1.0)
one_step_optimizer.step()

after = one_step_model.head[-1].weight.detach()
print("loss:", one_step_loss.item(), "gradient norm:", float(gradient_norm))
print("parameter changed:", not torch.equal(before, after))
assert not torch.equal(before, after)

In [ ]:
def run_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train(training)
    total_loss, predictions, targets = 0.0, [], []

    context = torch.enable_grad() if training else torch.inference_mode()
    with context:
        for batch in loader:
            batch = ACCELERATOR.move(batch)
            if training:
                optimizer.zero_grad(set_to_none=True)
            logits = model(batch["input_ids"], batch["attention_mask"])
            loss = nn.functional.cross_entropy(logits, batch["labels"])
            if training:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()

            total_loss += loss.item() * len(batch["labels"])
            predictions.extend(logits.argmax(-1).cpu().tolist())
            targets.extend(batch["labels"].cpu().tolist())

    return {
        "loss": total_loss / len(targets),
        "accuracy": sum(p == y for p, y in zip(predictions, targets)) / len(targets),
        "macro_f1": f1_score(targets, predictions, average="macro", zero_division=0),
    }

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    collate_fn=collate_tickets,
    generator=torch.Generator().manual_seed(SEED),
    num_workers=0,
)
valid_loader = DataLoader(
    valid_dataset,
    batch_size=32,
    shuffle=False,
    collate_fn=collate_tickets,
    num_workers=0,
)

torch.manual_seed(SEED)
model = MeanPoolClassifier(config).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-3, weight_decay=1e-4)
print("parameters:", sum(parameter.numel() for parameter in model.parameters()))

In [ ]:
history = []
best_state = None
best_f1 = -1.0

for epoch in range(1, 7):
    train_metrics = run_epoch(model, train_loader, optimizer)
    valid_metrics = run_epoch(model, valid_loader)
    history.append(
        {
            "epoch": epoch,
            **{f"train_{k}": v for k, v in train_metrics.items()},
            **{f"valid_{k}": v for k, v in valid_metrics.items()},
        }
    )
    if valid_metrics["macro_f1"] > best_f1:
        best_f1 = valid_metrics["macro_f1"]
        best_state = {
            name: value.detach().cpu().clone()
            for name, value in model.state_dict().items()
        }
    print(
        f"epoch={epoch} train_loss={train_metrics['loss']:.3f} "
        f"valid_loss={valid_metrics['loss']:.3f} "
        f"valid_f1={valid_metrics['macro_f1']:.3f}"
    )

history_df = pd.DataFrame(history)
history_df.round(3)

## 6. Checkpoint 저장

state dict만 저장하면 추론 시 vocabulary와 label 순서를 잃을 수 있습니다. 최소한 다음을 함께 저장합니다.

- model state와 모델 config
- token 순서(`itos`)와 label 순서
- seed, 학습 epoch, 검증 지표 같은 provenance

In [ ]:
CHECKPOINT_PATH = PROJECT_ROOT / "artifacts" / "notebook_text_classifier_exercise.pt"
CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
model.load_state_dict(best_state)

checkpoint = {
    "model_state": model.state_dict(),
    "model_config": asdict(config),
    "vocabulary": vocabulary.itos,
    "label_names": label_names,
    "seed": SEED,
    "best_validation_f1": best_f1,
}
torch.save(checkpoint, CHECKPOINT_PATH)
print("saved:", CHECKPOINT_PATH)
print("size (KB):", round(CHECKPOINT_PATH.stat().st_size / 1024, 1))

# 새 Python 프로세스에서도 같은 방식으로 복원할 수 있는지 확인합니다.
restored_payload = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=True)
restored_vocab = Vocabulary(restored_payload["vocabulary"])
restored_config = ClassifierConfig(**restored_payload["model_config"])
restored_model = MeanPoolClassifier(restored_config).to(DEVICE)
restored_model.load_state_dict(restored_payload["model_state"])
restored_model.eval()

assert restored_vocab.itos == vocabulary.itos
assert restored_payload["label_names"] == label_names
print("restored best F1:", restored_payload["best_validation_f1"])

In [ ]:
@torch.inference_mode()
def predict_text(text: str, model=restored_model, vocab=restored_vocab):
    ids = torch.tensor(vocab.encode(text), dtype=torch.long, device=DEVICE).unsqueeze(0)
    mask = ids.ne(vocab.stoi["<pad>"])
    probabilities = model(ids, mask).softmax(-1).squeeze(0)
    ranked = probabilities.argsort(descending=True)
    return [
        (label_names[index], float(probabilities[index]))
        for index in ranked.cpu().tolist()
    ]


for text in [
    "카드 결제가 두 번 되었어요",
    "로그인이 안 되고 비밀번호 재설정도 실패합니다",
    "API에서 429 오류가 납니다",
]:
    print("\n", text)
    print(predict_text(text)[:3])

## 7. Error analysis

전체 점수 하나만 보면 개선 방향을 알기 어렵습니다. 틀린 샘플의 정답, 예측, confidence, 텍스트를 한 테이블로 모읍니다. confidence가 높으면서 틀린 사례는 특히 우선적으로 봅니다.

In [ ]:
restored_model.eval()
error_rows = []
all_rows = []

with torch.inference_mode():
    for batch in valid_loader:
        batch = ACCELERATOR.move(batch)
        logits = restored_model(batch["input_ids"], batch["attention_mask"])
        probabilities = logits.softmax(-1)
        confidence, predicted_ids = probabilities.max(-1)
        for ticket_id, text, target_id, predicted_id, score in zip(
            batch["ticket_ids"],
            batch["texts"],
            batch["labels"],
            predicted_ids,
            confidence,
        ):
            row = {
                "ticket_id": ticket_id,
                "truth": id_to_label[int(target_id)],
                "prediction": id_to_label[int(predicted_id)],
                "confidence": float(score),
                "text": text,
            }
            all_rows.append(row)
            if row["truth"] != row["prediction"]:
                error_rows.append(row)

errors = pd.DataFrame(error_rows).sort_values("confidence", ascending=False)
print("validation rows:", len(all_rows), "errors:", len(errors))
errors.head(10)

predictions_df = pd.DataFrame(all_rows)
confusion = pd.crosstab(
    predictions_df["truth"],
    predictions_df["prediction"],
    rownames=["truth"],
    colnames=["prediction"],
    dropna=False,
).reindex(index=label_names, columns=label_names, fill_value=0)
confusion

## 8. 따라 치기 TODO

새 셀을 만들어 다음을 직접 구현하세요.

1. `min_frequency=2`로 vocabulary를 다시 만들고 OOV 비율과 F1을 비교합니다.
2. `mean pooling`을 `max pooling`으로 교체합니다. padding 위치는 매우 작은 값으로 가려야 합니다.
3. embedding 차원을 `16, 32, 64`로 바꾸고 parameter 수·학습 시간·F1을 표로 만듭니다.
4. label별 precision/recall을 직접 계산해 confusion table과 검산합니다.
5. 체크포인트에 데이터 파일의 SHA-256을 넣어 어떤 데이터로 학습했는지 추적합니다.

**도전 과제:** 같은 `case_id`를 무시한 무작위 행 분할을 만들어 점수가 얼마나 달라지는지 확인하고, 왜 그 점수를 신뢰하면 안 되는지 한 문단으로 설명하세요.

## 현업 체크리스트

- split 전에 중복·그룹·시간 순서를 확인한다.
- tokenizer/vocabulary는 train 데이터로만 fit한다.
- `Dataset`은 한 샘플, `collate_fn`은 배치 구성을 책임지게 한다.
- 모든 모델 경계에 shape와 dtype 계약을 둔다.
- validation에서는 `eval()`과 `inference_mode()`를 사용한다.
- checkpoint에 전처리 상태, label mapping, config, seed를 함께 저장한다.
- 평균 지표뿐 아니라 label별 지표와 실제 오류 문장을 검토한다.

다음 노트북에서는 PyTorch 모듈에 기대기 전에 MSE, linear/logistic regression, MLP의 핵심 계산을 더 낮은 수준에서 직접 구현합니다.